In [0]:
%run /Workspace/Users/superromelus@gmail.com/SalesLT/01_load_bronze

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — CLEANING RULE 1 : Trim + empty → NULL  (one pass, all strings)
# ─────────────────────────────────────────────────────────────────────────────
# WHY : Trim whitespace AND convert empty strings to NULL together in one SQL
#       pass — no need for two separate rules at this scale.
# HOW : NULLIF(TRIM(col), '') handles both in a single expression.
 
from pyspark.sql.types import StringType
 
for table, src in active_views.items():
    schema = spark.table(src).schema
    parts  = [
        f"NULLIF(TRIM(`{f.name}`), '') AS `{f.name}`"
        if isinstance(f.dataType, StringType) else f"`{f.name}`"
        for f in schema.fields
    ]
    new_view = f"c1_{table}"
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {new_view} AS
        SELECT {', '.join(parts)} FROM {src}
    """)
    active_views[table] = new_view
 
print("✓ RULE 1 — Trim + empty → NULL applied")
 

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — CLEANING RULE 2 : Standardize dates to yyyy-MM-dd
# ─────────────────────────────────────────────────────────────────────────────
# WHY : Date columns come as TIMESTAMP from the parquet files.
#       Casting to DATE string makes them safe for SQL comparisons.
# HOW : DATE_FORMAT(FROM_UTC_TIMESTAMP(col, 'UTC'), 'yyyy-MM-dd')
 
from pyspark.sql.types import TimestampType, DateType
 
for table, src in active_views.items():
    schema = spark.table(src).schema
    parts  = []
    for f in schema.fields:
        if isinstance(f.dataType, TimestampType):
            parts.append(
                f"DATE_FORMAT(FROM_UTC_TIMESTAMP(`{f.name}`, 'UTC'), 'yyyy-MM-dd') AS `{f.name}`"
            )
        elif isinstance(f.dataType, DateType):
            parts.append(f"DATE_FORMAT(`{f.name}`, 'yyyy-MM-dd') AS `{f.name}`")
        else:
            parts.append(f"`{f.name}`")
 
    new_view = f"c2_{table}"
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {new_view} AS
        SELECT {', '.join(parts)} FROM {src}
    """)
    active_views[table] = new_view
 
print("✓ RULE 2 — Date formatting applied")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — CLEANING RULE 3 : Standardize casing on key columns only
# ─────────────────────────────────────────────────────────────────────────────
# WHY : Casing only matters where values are used in filters/joins.
#       We target the highest-value columns only.
# HOW : SQL LOWER / INITCAP on specific columns per table.
 
CASING = {
    "customer":           {"EmailAddress": "lower",
                           "FirstName": "title", "LastName": "title"},
    "product":            {"Color": "title"},
    "sales_order_header": {},
    "sales_order_detail": {},
}
SQL_FN = {"lower": "LOWER", "upper": "UPPER", "title": "INITCAP"}
 
for table, src in active_views.items():
    schema     = spark.table(src).schema
    casing_map = CASING.get(table, {})
    # Map by snake_case name (columns already snake_case from bronze)
    snake_map  = {k.lower(): v for k, v in casing_map.items()}
    parts = []
    for f in schema.fields:
        mode = snake_map.get(f.name.lower())
        if mode:
            parts.append(f"{SQL_FN[mode]}(`{f.name}`) AS `{f.name}`")
        else:
            parts.append(f"`{f.name}`")
 
    new_view = f"c3_{table}"
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {new_view} AS
        SELECT {', '.join(parts)} FROM {src}
    """)
    active_views[table] = new_view
 
print("✓ RULE 3 — Casing standardized")